In [0]:
dbutils.library.restartPython()

In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, silver

cfg = common.load_config("dev")
silver.ensure_tables(spark, cfg)
tables = silver.load_silver()
want = {"hospitals": 5, "departments": 30, "insurance_claims": 81}

start = time.time()
second = []
for run in (1, 2):                                   # the second run must find nothing new
    for table in want:
        r = silver.load_latest(spark, cfg, table, tables[table])
        print(f"run {run} {table:18} {r}")
        if run == 2:
            second.append(r["rows"] + r["inserted"] + r["updated"])
got = {t: spark.table(f"{cfg['catalog']}.silver.{t}").count() for t in want}
print(got, f"in {time.time() - start:.0f}s")
print("PASS" if got == want and sum(second) == 0 else "FAIL")
display(spark.table(f"{cfg['catalog']}.silver.insurance_claims").limit(5))